# Gdg campus on Sookmyung - 1분기 Toy Project
### AI-ML Advanced - NLP / **인공지능공학부 이나연 - 'FastReact' 프로젝트**

### 1. Hugging Face 인증 및 Colab 환경 세팅

- **Gated Model 접근 승인:** Hugging Face의 `meta-llama/Llama-3.2-1B-Instruct` 저장소에서 Meta 라이선스 약관에 동의해 Llama 3.2 Gating Group 접근 승인을 획득합니다.
- **보안 토큰 연동:** `Read` 권한의 Access Token을 발급받아 Google Colab의 보안 비밀(`HF_TOKEN`)에 등록하고 `huggingface_hub.login()`으로 인증을 완료합니다. 로컬 VS Code 환경과 호환되도록 `try-except` 분기로 처리합니다.
- **GPU 추론 환경 구성:** Colab T4 GPU 런타임에서 `torch.float16` 정밀도와 `device_map="auto"` 설정으로 모델을 로드하여 약 2.4GB 수준의 VRAM 점유율 내에서 안정적으로 실행되는 것을 확인했습니다.

In [ ]:
import os
import re
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

# 1. Hugging Face 인증 (Colab & 로컬 환경 호환)
try:
   # Google colab인 경우
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    # 로컬 VS Code 환경일 때 (.env 또는 환경변수에서 로드)
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token)

# 2. 모델 및 토크나이저 로드 (디버깅용 1B / 벤치마크용 3B 전환 가능)
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print(f"로드 완료: {MODEL_ID} | Device: {model.device}")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

로드 완료: meta-llama/Llama-3.2-1B-Instruct | Device: cuda:0


### 2. Mock 도구 및 System Prompt 정의

- **Mock 도구:** 외부 네트워크 지연을 배제하고 순수 추론 속도를 측정하기 위해 로컬 문서 검색(`search`)과 사칙연산(`calculator`) 파이썬 함수를 구현했습니다.
- **System Prompt:** 모델이 `Thought → Action → Observation` 형식을 준수하고, 충분한 정보를 얻었을 때 `Final Answer`로 루프를 종료하도록 지시문과 Few-shot 예시를 배치했습니다.

In [ ]:
# 1. Mock 문서 DB (추후 HotpotQA 지원 문서나 위키 데이터로 교체 가능, 일단 더미 데이터로 확인)
MOCK_DOCS = {
    "inception": "Inception is a 2010 science fiction action film written and directed by Christopher Nolan. Its runtime is 148 minutes.",
    "interstellar": "Interstellar is a 2014 epic science fiction film directed by Christopher Nolan. Its runtime is 169 minutes.",
    "christopher nolan": "Christopher Nolan is a British and American filmmaker born on 30 July 1970."
}

def search(query: str) -> str:
    """키워드 토큰 기반 로컬 문서 검색 Mock 함수 (제목 + 본문 검색)"""
    query_words = set(re.findall(r"\w+", query.lower()))
    # 의미 없는 불용어 제외
    stop_words = {"the", "a", "an", "of", "in", "is", "what", "and", "to"}
    keywords = query_words - stop_words

    best_match = None
    max_hits = 0

    for key, content in MOCK_DOCS.items():
        doc_text = (key + " " + content).lower()
        hits = sum(1 for kw in keywords if kw in doc_text)
        if hits > max_hits:
            max_hits = hits
            best_match = content

    if best_match and max_hits > 0:
        return best_match
    return f"No relevant documents found for '{query}'."

def calculator(expression: str) -> str:
    # 사칙연산 계산기 Mock 함수
    # 숫자와 기본 연산자만 허용
    if not re.match(r"^[0-9+\-*/().\s]+$", expression):
        return "Error: Invalid mathematical expression."
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

TOOLS = {
    "search": search,
    "calculator": calculator
}

# 2. ReAct 시스템 프롬프트 (고정 Prefix 영역)
SYSTEM_PROMPT = """You are a helpful AI assistant that solves problems step-by-step using tools.
You have access to the following tools:
- search[query]: Search for factual information in the knowledge base.
- calculator[expression]: Evaluate a mathematical expression (e.g., 148 + 169).

You MUST strictly follow this format in every turn:
Thought: <your reasoning about what to do next>
Action: <tool_name>[<tool_input>]

When you receive an Observation, continue with a new Thought.
Once you have enough information to answer the user's question, output:
Thought: <final reasoning>
Final Answer: <your final answer>

Example:
Question: What is the sum of X and Y?
Thought: I need to find the value of X first.
Action: search[X]
Observation: X is 10.
Thought: Now I need to find the value of Y.
Action: search[Y]
Observation: Y is 25.
Thought: Now I can add 10 and 25.
Action: calculator[10 + 25]
Observation: 35
Thought: I have the final answer.
Final Answer: 35

Rules:
- Output ONLY ONE Action per turn, then stop and wait for the Observation.
- Do NOT generate the Observation yourself.
"""

### 3. 바닐라 ReAct 루프 및 측정 로직
- 매 턴 누적된 `messages` 전체를 `apply_chat_template()`으로 다시 포맷팅하고 처음부터 토크나이징하여 `model.generate()`에 입력합니다.
- 추후 KV Cache 버전과의 출력 동일성(토큰 일치율) 검증을 위해 `do_sample=False` 기반의 Greedy 디코딩을 고정합니다.
- 모델이 스스로 가짜 `Observation:`을 생성하는 환각을 막기 위해 `stop_strings=["Observation:"]`로 생성을 중단시킨 뒤, 정규표현식(`parse_action`)으로 툴 이름과 인자를 파싱해 실행합니다.
- `time.perf_counter()`로 턴별 Latency를, `torch.cuda.max_memory_allocated()`로 Peak VRAM을 기록합니다.

In [ ]:
# 바닐라 ReAct 루프를 구현합니다.
def parse_action(text: str):
    # 모델 출력에서 Action: tool_name[tool_input] 또는 Final Answer 추출
    final_match = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)
    if final_match:
        return "final", final_match.group(1).strip()

    action_match = re.search(r"Action:\s*([a-zA-Z0-9_]+)\[([^\]]+)\]", text)
    if action_match:
        return action_match.group(1).strip(), action_match.group(2).strip()

    return None, None


def run_vanilla_react(question: str, max_turns: int = 5):
    # VRAM 피크 측정 초기화
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # 대화 내역 (매 턴 전체를 다시 토크나이징할 대상)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Question: {question}"}
    ]

    turn_metrics = []

    for turn in range(1, max_turns + 1):
        print(f"\n--- [Turn {turn}] ---")

        # 1. 매 턴 전체 컨텍스트를 처음부터 다시 문자열화 및 토크나이징 (Vanilla의 비효율 지점)
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]

        # 2. 모델 추론 (Greedy 디코딩 고정 + "Observation:" 생성 시 중단)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = time.perf_counter()

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.pad_token_id,
                stop_strings=["Observation:"],
                tokenizer=tokenizer
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start_time

        # 새로 생성된 토큰만 디코딩
        generated_ids = outputs[0][input_len:]
        response_text = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
            ).strip()

        # stop_strings로 인해 끝에 붙을 수 있는 "Observation:" 잔여물 제거
        response_text = response_text.replace("Observation:", "").strip()

        print(f"[Input Tokens: {input_len} | Latency: {elapsed:.4f}s]")
        print(response_text)

        turn_metrics.append({
            "turn": turn,
            "input_tokens": input_len,
            "latency": elapsed
        })

        # 3. Action 파싱 및 툴 실행
        tool_name, tool_input = parse_action(response_text)

        if tool_name == "final":
            print(f"\n 최종 답변 도출 완료: {tool_input}")
            break
        elif tool_name in TOOLS:
            observation = TOOLS[tool_name](tool_input)
            print(f"Observation: {observation}")

            # Assistant의 출력과 Tool의 Observation을 대화 이력에 추가
            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        else:
            # 파싱 실패 시 재유도 (추후 롤백 실험 대상이 되는 지점)
            print("Warning: Action 파싱 실패, 재시도 유도")
            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "user", "content": "Observation: Invalid format. Use 'Action: tool_name[input]' or 'Final Answer: ...'"})

    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0
    print(f"\n[Summary] Total Turns: {len(turn_metrics)} | Peak VRAM: {peak_vram_mb:.2f} MB")
    return turn_metrics, peak_vram_mb

### 4. 실행 및 검증
- 다중 단계 추론이 필요한 복합 질문("Inception과 Interstellar의 총 러닝타임 합계")을 입력해 멀티턴 누적과 도구 호출 흐름을 검증합니다.

In [ ]:
test_question = "What is the combined runtime in minutes of Inception and Interstellar?"
metrics, peak_vram = run_vanilla_react(test_question)


--- [Turn 1] ---
[Input Tokens: 310 | Latency: 2.9925s]
Thought: I need to find the runtime of Inception and Interstellar.
Action: search[runtime]
Observation: Inception is a 2010 science fiction action film written and directed by Christopher Nolan. Its runtime is 148 minutes.

--- [Turn 2] ---
[Input Tokens: 369 | Latency: 2.9109s]
Thought: Now I need to find the runtime of Interstellar.
Action: search[runtime]
Observation: Inception is a 2010 science fiction action film written and directed by Christopher Nolan. Its runtime is 148 minutes.

--- [Turn 3] ---
[Input Tokens: 426 | Latency: 1.9662s]
Thought: Now I need to find the combined runtime of Inception and Interstellar.
Action: calculator[148 + 178]
Observation: 326

--- [Turn 4] ---
[Input Tokens: 467 | Latency: 1.2706s]
Thought: I have the combined runtime of Inception and Interstellar.
Action: None

--- [Turn 5] ---
[Input Tokens: 515 | Latency: 1.4253s]
Thought: I need to find the combined runtime of Inception and Interstel